# P5: ML ROI Dashboard & Presentation

**Module 3 — Week 14**

**Problem Statement:** Quantify the business value of the churn prediction model and build a stakeholder-facing monitoring dashboard.

**Success Metrics:**
- Positive ROI demonstrated with cost-benefit analysis
- Comprehensive production monitoring covering model, system, data, and business metrics
- Clear executive summary suitable for non-technical stakeholders

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
def generate_production_metrics(n_days=90):
    """Generate 90 days of simulated production metrics.
    
    Includes:
    - AUC: starts at 0.85, gradual degradation after day 60
    - Latency p50/p95: ~50ms baseline, spike on day 45
    - Daily requests: weekly seasonal pattern
    - Error rate: mostly low with occasional spikes
    - Feature drift: monthly_charges mean shifts gradually
    """
    np.random.seed(42)
    dates = [datetime.now() - timedelta(days=n_days - i) for i in range(n_days)]
    
    # Model AUC: gradual degradation after day 60
    base_auc = 0.85
    auc = [base_auc + np.random.normal(0, 0.005) - max(0, (i - 60) * 0.002)
           for i in range(n_days)]
    
    # Latency: ~50ms normally, spike on days 44-46
    latency_p50 = [50 + np.random.normal(0, 5) + (100 if 44 <= i <= 46 else 0)
                   for i in range(n_days)]
    latency_p95 = [p * 2.5 + np.random.normal(0, 10) for p in latency_p50]
    
    # Request volume: weekly seasonal pattern
    requests = [int(1000 + 200 * np.sin(i * 2 * np.pi / 7) + np.random.normal(0, 50))
                for i in range(n_days)]
    
    # Error rate: mostly low, occasional spikes
    error_rate = [0.001 + np.random.exponential(0.0005) for _ in range(n_days)]
    
    # Feature drift: monthly_charges mean shifts gradually
    monthly_charges_mean = [65 + i * 0.15 + np.random.normal(0, 5) for i in range(n_days)]
    tenure_mean = [32 + np.random.normal(0, 3) for _ in range(n_days)]
    
    return pd.DataFrame({
        'date': dates,
        'auc': auc,
        'latency_p50': latency_p50,
        'latency_p95': latency_p95,
        'requests': requests,
        'error_rate': error_rate,
        'monthly_charges_mean': monthly_charges_mean,
        'tenure_mean': tenure_mean,
    })

## 2. Generate Production Metrics

In [ ]:
metrics = generate_production_metrics(n_days=90)
print(f"Generated {len(metrics)} days of production metrics")
print(f"Date range: {metrics['date'].iloc[0]:%Y-%m-%d} to {metrics['date'].iloc[-1]:%Y-%m-%d}")
print()
print(metrics.describe().round(3))

In [ ]:
metrics.head(10)

## 3. Model Performance Monitoring

**TODO:** Create a 4-panel matplotlib dashboard showing:
1. AUC over time with threshold line at 0.80 and annotation for degradation start (day 60)
2. Latency p50 and p95 over time with SLA line at 200ms
3. Daily request volume with 7-day moving average
4. Error rate with alert threshold

In [ ]:
# TODO: Build 4-panel monitoring dashboard
# Hints:
# - Use fig, axes = plt.subplots(2, 2, figsize=(16, 10))
# - Panel 1: AUC with ax.axhline(y=0.80) and ax.annotate() for degradation
# - Panel 2: Latency p50 + p95 with SLA line
# - Panel 3: Requests bar chart + rolling(7).mean() overlay
# - Panel 4: Error rate with alert threshold line

# ============================================================
# SOLUTION (uncomment to use)
# ============================================================
# fig, axes = plt.subplots(2, 2, figsize=(16, 10))
#
# # Panel 1: AUC
# ax = axes[0, 0]
# ax.plot(metrics['date'], metrics['auc'], color='#2ecc71', linewidth=1.5)
# ax.axhline(y=0.80, color='red', linestyle='--', alpha=0.5, label='Threshold (0.80)')
# ax.annotate('Degradation starts', xy=(metrics['date'].iloc[60], metrics['auc'].iloc[60]),
#             xytext=(metrics['date'].iloc[45], 0.82),
#             arrowprops=dict(arrowstyle='->', color='red'),
#             fontsize=10, color='red')
# ax.set_ylabel('AUC')
# ax.set_title('Model AUC Over Time')
# ax.legend()
# ax.tick_params(axis='x', rotation=45)
#
# # Panel 2: Latency
# ax = axes[0, 1]
# ax.plot(metrics['date'], metrics['latency_p50'], color='#3498db', label='p50', linewidth=1.5)
# ax.plot(metrics['date'], metrics['latency_p95'], color='#e74c3c', label='p95', linewidth=1.5)
# ax.axhline(y=200, color='red', linestyle='--', alpha=0.5, label='SLA (200ms)')
# ax.set_ylabel('Latency (ms)')
# ax.set_title('API Latency')
# ax.legend()
# ax.tick_params(axis='x', rotation=45)
#
# # Panel 3: Request Volume
# ax = axes[1, 0]
# ax.bar(metrics['date'], metrics['requests'], alpha=0.3, color='#3498db', label='Daily')
# rolling_avg = metrics['requests'].rolling(7).mean()
# ax.plot(metrics['date'], rolling_avg, color='#2c3e50', linewidth=2, label='7-day MA')
# ax.set_ylabel('Requests')
# ax.set_title('Daily Request Volume')
# ax.legend()
# ax.tick_params(axis='x', rotation=45)
#
# # Panel 4: Error Rate
# ax = axes[1, 1]
# ax.plot(metrics['date'], metrics['error_rate'], color='#e74c3c', linewidth=1.5)
# ax.axhline(y=0.005, color='orange', linestyle='--', alpha=0.7, label='Alert threshold')
# ax.set_ylabel('Error Rate')
# ax.set_title('Error Rate')
# ax.legend()
# ax.tick_params(axis='x', rotation=45)
#
# plt.suptitle('Production Monitoring Dashboard', fontsize=14, fontweight='bold')
# plt.tight_layout()
# plt.savefig('outputs/monitoring_dashboard.png', dpi=150, bbox_inches='tight')
# plt.show()

## 4. Drift Analysis

**TODO:**
1. Implement `compute_psi(reference, current, bins=10)` — Population Stability Index
2. Split the 90 days into reference (first 30 days) vs 3 current periods (31-50, 51-70, 71-90)
3. Compute PSI for `monthly_charges_mean` and `tenure_mean` in each period
4. Plot PSI over periods with threshold lines at 0.1 (moderate) and 0.25 (significant)
5. Identify which features are drifting and when

In [ ]:
# TODO: Implement PSI and drift analysis
# Hints:
# - PSI = sum((actual_pct - expected_pct) * ln(actual_pct / expected_pct))
# - Use np.histogram to bin the distributions
# - Replace zeros with small epsilon to avoid log(0)
# - PSI < 0.1: no significant drift
# - 0.1 <= PSI < 0.25: moderate drift
# - PSI >= 0.25: significant drift

# ============================================================
# SOLUTION (uncomment to use)
# ============================================================
# def compute_psi(reference, current, bins=10):
#     """Compute Population Stability Index between reference and current distributions."""
#     eps = 1e-4
#     # Use reference bins for both
#     breakpoints = np.linspace(min(reference.min(), current.min()),
#                               max(reference.max(), current.max()), bins + 1)
#     ref_counts = np.histogram(reference, bins=breakpoints)[0]
#     cur_counts = np.histogram(current, bins=breakpoints)[0]
#     
#     ref_pct = ref_counts / len(reference) + eps
#     cur_pct = cur_counts / len(current) + eps
#     
#     psi = np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct))
#     return psi
#
# # Define periods
# reference = metrics.iloc[:30]
# periods = [
#     ('Days 31-50', metrics.iloc[30:50]),
#     ('Days 51-70', metrics.iloc[50:70]),
#     ('Days 71-90', metrics.iloc[70:90]),
# ]
#
# features = ['monthly_charges_mean', 'tenure_mean']
# psi_results = {}
#
# for feat in features:
#     psi_results[feat] = []
#     for period_name, period_data in periods:
#         psi_val = compute_psi(reference[feat].values, period_data[feat].values)
#         psi_results[feat].append(psi_val)
#         print(f"{feat} | {period_name}: PSI = {psi_val:.4f}")
#
# # Plot PSI
# fig, ax = plt.subplots(figsize=(8, 5))
# period_labels = [p[0] for p in periods]
# x = np.arange(len(period_labels))
# width = 0.35
#
# ax.bar(x - width/2, psi_results['monthly_charges_mean'], width, label='monthly_charges', color='#3498db')
# ax.bar(x + width/2, psi_results['tenure_mean'], width, label='tenure', color='#2ecc71')
# ax.axhline(y=0.1, color='orange', linestyle='--', label='Moderate drift (0.1)')
# ax.axhline(y=0.25, color='red', linestyle='--', label='Significant drift (0.25)')
# ax.set_xticks(x)
# ax.set_xticklabels(period_labels)
# ax.set_ylabel('PSI')
# ax.set_title('Feature Drift (PSI) Over Time')
# ax.legend()
# plt.tight_layout()
# plt.show()
#
# print("\nDrift Summary:")
# for feat in features:
#     max_psi = max(psi_results[feat])
#     if max_psi >= 0.25:
#         print(f"  {feat}: SIGNIFICANT drift detected (PSI={max_psi:.4f})")
#     elif max_psi >= 0.1:
#         print(f"  {feat}: Moderate drift detected (PSI={max_psi:.4f})")
#     else:
#         print(f"  {feat}: No significant drift (PSI={max_psi:.4f})")

## 5. ROI Calculation

**TODO:** Calculate the ROI of the churn prediction model.

**Scenario:**
- 5,000 customers, 25% annual churn rate, $1,200 customer LTV, $50 retention offer cost
- **Without model:** email all customers, cost = 5000 x $50, save rate ~15%
- **With model** (precision=0.72, recall=0.65): targeted approach, 40% save rate on true positives

Calculate and compare: targeted customers, true positives, saved customers, value, cost, net value, ROI%

In [ ]:
# TODO: Implement ROI comparison
# Hints:
# - No-model: target everyone, low save rate (15%)
# - With-model: target predicted churners, higher save rate (40%) on true positives
# - actual_churners = n_customers * churn_rate
# - predicted_churners = actual_churners * recall / precision
# - true_positives = predicted_churners * precision
# - Infrastructure cost: $500/month

# ============================================================
# SOLUTION (uncomment to use)
# ============================================================
# n_customers = 5000
# churn_rate = 0.25
# customer_ltv = 1200
# retention_cost = 50
# infra_monthly = 500
# precision = 0.72
# recall = 0.65
#
# actual_churners = int(n_customers * churn_rate)
#
# # --- Without Model ---
# no_model_targeted = n_customers  # email everyone
# no_model_cost = no_model_targeted * retention_cost
# no_model_save_rate = 0.15
# no_model_saved = int(actual_churners * no_model_save_rate)
# no_model_value = no_model_saved * customer_ltv
# no_model_net = no_model_value - no_model_cost
# no_model_roi = (no_model_net / no_model_cost) * 100
#
# # --- With Model ---
# predicted_churners = int(actual_churners * recall / precision)
# true_positives = int(predicted_churners * precision)
# model_save_rate = 0.40
# model_saved = int(true_positives * model_save_rate)
# model_cost = predicted_churners * retention_cost + infra_monthly * 12
# model_value = model_saved * customer_ltv
# model_net = model_value - model_cost
# model_roi = (model_net / model_cost) * 100
#
# # --- Comparison ---
# comparison = pd.DataFrame({
#     'Metric': ['Targeted Customers', 'True Positives', 'Saved Customers',
#                'Annual Value ($)', 'Annual Cost ($)', 'Net Value ($)', 'ROI (%)'],
#     'No Model': [no_model_targeted, actual_churners, no_model_saved,
#                  no_model_value, no_model_cost, no_model_net, f"{no_model_roi:.0f}%"],
#     'With Model': [predicted_churners, true_positives, model_saved,
#                    model_value, model_cost, model_net, f"{model_roi:.0f}%"],
# })
# print(comparison.to_string(index=False))
# print(f"\nIncremental value of ML model: ${model_net - no_model_net:,.0f}")

## 6. Sensitivity Analysis

**TODO:**
1. Vary each parameter independently while holding others at baseline
   - Precision: 0.4 to 0.9
   - Recall: 0.4 to 0.9
   - Retention cost: $20 to $100
   - Customer LTV: $500 to $2,000
2. Compute ROI for each parameter value
3. Create 4-panel plot showing ROI vs each parameter
4. Identify break-even points where ROI = 0

In [ ]:
# TODO: Implement sensitivity analysis
# Hints:
# - Reuse the ROI calculation logic from section 5
# - Use np.linspace to generate parameter ranges
# - For break-even: find where ROI crosses zero (np.interp)

# ============================================================
# SOLUTION (uncomment to use)
# ============================================================
# def calc_roi(n_cust, churn, prec, rec, ret_cost, ltv, infra):
#     """Helper to compute ROI for sensitivity analysis."""
#     actual = int(n_cust * churn)
#     predicted = int(actual * rec / max(prec, 0.01))
#     tp = int(predicted * prec)
#     saved = int(tp * 0.40)
#     value = saved * ltv
#     cost = predicted * ret_cost + infra * 12
#     net = value - cost
#     return (net / cost * 100) if cost > 0 else 0
#
# # Baseline parameters
# base = dict(n_cust=5000, churn=0.25, prec=0.72, rec=0.65,
#             ret_cost=50, ltv=1200, infra=500)
#
# # Parameter sweeps
# sweeps = {
#     'Precision': ('prec', np.linspace(0.4, 0.9, 20)),
#     'Recall': ('rec', np.linspace(0.4, 0.9, 20)),
#     'Retention Cost ($)': ('ret_cost', np.linspace(20, 100, 20)),
#     'Customer LTV ($)': ('ltv', np.linspace(500, 2000, 20)),
# }
#
# fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# for ax, (title, (param, values)) in zip(axes.flatten(), sweeps.items()):
#     rois = []
#     for v in values:
#         params = base.copy()
#         params[param] = v
#         rois.append(calc_roi(**params))
#     
#     ax.plot(values, rois, color='#3498db', linewidth=2)
#     ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Break-even')
#     ax.fill_between(values, rois, 0, where=[r > 0 for r in rois],
#                     alpha=0.1, color='green')
#     ax.fill_between(values, rois, 0, where=[r <= 0 for r in rois],
#                     alpha=0.1, color='red')
#     ax.set_xlabel(title)
#     ax.set_ylabel('ROI (%)')
#     ax.set_title(f'ROI vs {title}')
#     ax.legend()
#     
#     # Find break-even point
#     for i in range(len(rois) - 1):
#         if rois[i] * rois[i+1] < 0:  # sign change
#             be = np.interp(0, [rois[i], rois[i+1]], [values[i], values[i+1]])
#             ax.axvline(x=be, color='orange', linestyle=':', alpha=0.7)
#             ax.annotate(f'BE={be:.1f}', xy=(be, 0), fontsize=9, color='orange')
#
# plt.suptitle('Sensitivity Analysis: ROI vs Key Parameters', fontsize=14, fontweight='bold')
# plt.tight_layout()
# plt.savefig('outputs/sensitivity_analysis.png', dpi=150, bbox_inches='tight')
# plt.show()

## 7. Build Streamlit Dashboard

The interactive Streamlit dashboard is located at `src/dashboard.py`. It provides:

1. **KPI Row** — Current AUC, latency, request volume, and error rate with delta indicators
2. **Performance Charts** — AUC trend with threshold line and latency p50/p95 with SLA
3. **ROI Calculator** — Interactive sidebar controls for all business parameters
4. **ROI Metrics** — Saved customers, annual value, cost, and ROI percentage

### Running the Dashboard

```bash
streamlit run src/dashboard.py
```

In [ ]:
# Display the dashboard source code for reference
with open('src/dashboard.py', 'r') as f:
    print(f.read())

## 8. Executive Summary

**TODO:** Fill in the executive summary template below with values from your analysis.

---

## Executive Summary: Churn Prediction Model

### Problem
- Customer churn costs approximately $____ annually
- Previous approach: ____ (mass email, no targeting)

### Solution
- ML model predicts churn with ____% precision, ____% recall
- Enables targeted retention interventions

### Results (90-Day Production)
- Model AUC: ____ (current) vs ____ (baseline)
- API latency: ____ms p50, ____ms p95
- Uptime: ____%

### Business Impact
- Customers saved: ____ per year
- Revenue retained: $____
- Net ROI: ____%
- Payback period: ____ months

### Risks and Recommendations
1. Model degradation detected after day 60 — retraining scheduled
2. Feature drift in monthly_charges — investigate data pipeline
3. Recommend: automated retraining when PSI > 0.25

## 9. What I Built / What I Learned

### Deliverables
- [ ] 90-day production metrics simulation
- [ ] 4-panel monitoring dashboard
- [ ] Drift analysis with PSI
- [ ] ROI cost-benefit analysis (No Model vs With Model)
- [ ] Sensitivity analysis on key business parameters
- [ ] Interactive Streamlit dashboard
- [ ] Executive summary

### Key Learnings

**TODO:** Fill in after completing the project.

1. **ROI Calculation:** ...
2. **Production Monitoring:** ...
3. **Drift Detection:** ...
4. **Stakeholder Communication:** ...
5. **Dashboard Design:** ...